        # ⛺ P3　營地任務：特徵工程與洩漏陷阱
        **統計冒險之旅 2026**　｜　Day 3（09/24 四）🗻 預測之巔　｜　關卡　｜　🏅 100 XP

        📖 實作；資料：勇者咖啡會員 ＋ P1 的特徵表
        　分數好到不像話，就要懷疑

        ### 🎯 這一關你會學到
        - 逐步加入 RFM、分群標籤、交互作用，看交叉驗證 AUC 怎麼變
- 親手製造一次資料洩漏（AUC 0.99），再把它抓出來
- 分數好到不像話就要懷疑

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.2.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "P3"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["P3-1", "P3-2", "P3-3", "P3-4", "P3-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""

class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_P3_1(run):
    out, ns = run()
    return (約等於(抓變數(ns, "基準AUC"), 0.81798, 0.02), "基準AUC = 評分(X0, y)。")
任務定義("P3-1", _check_P3_1, 提示="評分(X0, y)。")

def _check_P3_2(run):
    out, ns = run()
    x = 抓變數(ns, "X1")
    ok, msg = 資料框像(x, 列=2000, 含欄位=["R", "F", "M"])
    if not ok: return (False, msg)
    return (約等於(抓變數(ns, "加RFM_AUC"), 0.81713, 0.02), "加RFM_AUC = 評分(X1, y)。")
任務定義("P3-2", _check_P3_2, 提示="評分(X1, y)。")

def _check_P3_3(run):
    out, ns = run()
    x = 抓變數(ns, "X2")
    ok, msg = 資料框像(x, 列=2000, 欄=19)
    if not ok: return (False, msg + " 群標籤 one-hot 後應多 4 欄。")
    return (約等於(抓變數(ns, "加分群_AUC"), 0.81647, 0.02), "加分群_AUC = 評分(X2, y)。")
任務定義("P3-3", _check_P3_3, 提示="評分(X2, y)。")

def _check_P3_4(run):
    out, ns = run()
    x = 抓變數(ns, "X3")
    ok, msg = 資料框像(x, 列=2000, 含欄位=["消費力", "活躍度"])
    if not ok: return (False, msg)
    if not 約等於(x["活躍度"].mean(), 0.97811, 相對=0.02): return (False, "活躍度 = 來店次數 / 加入月數。")
    return (約等於(抓變數(ns, "加交互_AUC"), 0.81640, 0.02), "加交互_AUC = 評分(X3, y)。")
任務定義("P3-4", _check_P3_4, 提示="members['來店次數'] / members['加入月數']。")

def _check_P3_5(run):
    out, ns = run()
    if not (抓變數(ns, "洩漏AUC") > 0.95): return (False, "洩漏AUC 應該高得不像話（> 0.95）；收到回購禮 要用 rng = np.random.default_rng(42)。")
    return (str(抓變數(ns, "洩漏特徵")) == "收到回購禮", "洩漏特徵 = 重要性最高的欄位名稱（重要性.index[0]）。")
任務定義("P3-5", _check_P3_5, 提示="重要性.index[0]。")


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0-rc.1/data/coffee_members.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0-rc.1/data/coffee_sales_aug.csv")
sales["日期"] = pd.to_datetime(sales["日期"])
print(members.shape, sales.shape)

## ⛺ 營地任務：特徵工程與洩漏陷阱
同一個模型、同一份資料，**換一組特徵，分數可以差很多**。這一關用「一條固定的評分流程」（Pipeline 標準化 → 邏輯斯迴歸 → 5 摺交叉驗證 AUC）當秤，逐步加入特徵看分數怎麼變：

1. 原始會員欄位（基準）
2. ＋ P1 的 RFM
3. ＋ K-means 分群標籤
4. ＋ 交互作用（消費力、活躍度）
5. ＋ 一個「偷看答案」的特徵——AUC 會飆到 0.99。然後我們把它抓出來。

> 🧭 秤要固定：每次都用同一個 `評分()` 函式，才能公平比較。

In [ ]:
def 評分(X, y):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    return float(cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean())
y = members["回購"]
X0 = pd.get_dummies(members.drop(columns=["會員編號", "回購"]), drop_first=True).astype(float)
print(X0.shape, "基準 AUC：", round(評分(X0, y), 3))

### 🎯 任務 P3-1　基準分數

用 `評分()` 算原始特徵 `X0` 的 5 摺交叉驗證 AUC，存成 `基準AUC`。

**預期結果（範例）**
```
0.818
```

In [ ]:
# 🎯 任務 P3-1　基準分數（請保留這一行）
基準AUC = ???
print(round(基準AUC, 3))

In [ ]:
檢查("P3-1")   # ◀ 執行這一格，看看任務 P3-1 有沒有過關

## 3-2　加入 RFM
P1 算過的 RFM 在這裡重算一次（用乾淨版 `sales`），接回會員表：沒來的人 F、M 填 0、R 填 31。

In [ ]:
基準日 = pd.Timestamp("2026-08-31")
rfm = sales.dropna(subset=["會員編號"]).groupby("會員編號").agg(
    R=("日期", lambda s: (基準日 - s.max()).days), F=("訂單編號", "count"), M=("金額", "sum")).reset_index()
mr = members[["會員編號"]].merge(rfm, on="會員編號", how="left")
mr["F"] = mr["F"].fillna(0); mr["M"] = mr["M"].fillna(0); mr["R"] = mr["R"].fillna(31)
print(mr.head())

### 🎯 任務 P3-2　＋RFM

把 `mr` 的 R、F、M 三欄接到 `X0` 右邊（`pd.concat(axis=1)`，列的順序本來就對齊）做成 `X1`，算 `加RFM_AUC`。

**預期結果（範例）**
```
(2000, 15) 0.817 比基準 低
```

In [ ]:
# 🎯 任務 P3-2　＋RFM（請保留這一行）
X1 = pd.concat([X0, mr[["R", "F", "M"]]], axis=1)
加RFM_AUC = ???
print(X1.shape, round(加RFM_AUC, 3), "比基準", "高" if 加RFM_AUC > 基準AUC else "低")

In [ ]:
檢查("P3-2")   # ◀ 執行這一格，看看任務 P3-2 有沒有過關

## 3-3　分群標籤當特徵
L09 的 K-means 可以反過來當特徵工程：先把會員分成 4 群，再把「第幾群」one-hot 後餵給分類模型。

### 🎯 任務 P3-3　＋分群標籤

對數值欄 `年齡、加入月數、來店次數、平均消費、距上次來店天數、住家距離` 標準化後做 `KMeans(n_clusters=4, n_init=10, random_state=42)`，把群標籤 one-hot（4 欄）接到 `X1` 做成 `X2`，算 `加分群_AUC`。

**預期結果（範例）**
```
(2000, 19) 0.816
```

In [ ]:
# 🎯 任務 P3-3　＋分群標籤（請保留這一行）
數值欄 = ["年齡", "加入月數", "來店次數", "平均消費", "距上次來店天數", "住家距離"]
Z = StandardScaler().fit_transform(members[數值欄])
群 = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Z)
群欄 = pd.get_dummies(pd.Series(群).astype(str), prefix="群").astype(float)
X2 = pd.concat([X1, 群欄], axis=1)
加分群_AUC = ???
print(X2.shape, round(加分群_AUC, 3))

In [ ]:
檢查("P3-3")   # ◀ 執行這一格，看看任務 P3-3 有沒有過關

## 3-4　交互作用：把兩個欄位乘起來
「平均消費 × 來店次數」= 這個人一個月大概貢獻多少（消費力）；「來店次數 ÷ 加入月數」= 活躍度。線性模型自己不會乘，你得幫它。

### 🎯 任務 P3-4　＋交互作用

在 `X2` 上新增 `消費力 = 平均消費 × 來店次數`、`活躍度 = 來店次數 ÷ 加入月數` 兩欄做成 `X3`，算 `加交互_AUC`。

**預期結果（範例）**
```
(2000, 21) 0.816
```

In [ ]:
# 🎯 任務 P3-4　＋交互作用（請保留這一行）
X3 = X2.copy()
X3["消費力"] = members["平均消費"] * members["來店次數"]
X3["活躍度"] = ???
加交互_AUC = 評分(X3, y)
print(X3.shape, round(加交互_AUC, 3))

In [ ]:
檢查("P3-4")   # ◀ 執行這一格，看看任務 P3-4 有沒有過關

## 3-5　親手製造一次資料洩漏
行銷部有一欄「收到回購禮」：**回購的人都會收到回購禮**（外加 5% 隨機贈送）。把它當特徵，AUC 會飆到 0.99——但這欄是在「回購發生之後」才填的，預測時根本拿不到。這叫**資料洩漏（leakage）**。

抓法有兩招：(1) 分數好到不像話；(2) 看特徵重要性——一個特徵一枝獨秀，就去查它是怎麼產生的、預測當下拿不拿得到。

In [ ]:
rng = np.random.default_rng(42)
收到回購禮 = ((members["回購"] == 1) | (rng.random(len(members)) < 0.05)).astype(int)
print(pd.crosstab(收到回購禮, members["回購"]))

### 🎯 任務 P3-5　洩漏：製造、發現、指認

把 `收到回購禮` 接到 `X3` 做成 `X4`，算 `洩漏AUC`；再用 `RandomForestClassifier(200, random_state=42)` fit 整份 `X4`，取特徵重要性最高的欄位名稱存成 `洩漏特徵`。

**預期結果（範例）**
```
0.992 收到回購禮
```

In [ ]:
# 🎯 任務 P3-5　洩漏：製造、發現、指認（請保留這一行）
rng = np.random.default_rng(42)
收到回購禮 = ((members["回購"] == 1) | (rng.random(len(members)) < 0.05)).astype(int)
X4 = X3.copy(); X4["收到回購禮"] = 收到回購禮
洩漏AUC = 評分(X4, y)
rf = RandomForestClassifier(200, random_state=42).fit(X4, y)
重要性 = pd.Series(rf.feature_importances_, index=X4.columns).sort_values(ascending=False)
洩漏特徵 = ???
print(round(洩漏AUC, 3), 洩漏特徵)
print(重要性.head(5).round(3))

In [ ]:
檢查("P3-5")   # ◀ 執行這一格，看看任務 P3-5 有沒有過關

## 3-6　把分數排起來看
| 特徵組合 | 交叉驗證 AUC |
|---|---|
| 原始欄位 | 0.818 |
| ＋RFM | 0.817 |
| ＋分群 | 0.816 |
| ＋交互作用 | 0.816 |
| ＋洩漏特徵 | 0.992 ⚠️ |

真正的特徵工程進步通常是 0.01～0.03 這種幅度；一跳 0.2 以上，先懷疑洩漏。

## 🌟 進階挑戰（不計分）
1. 用 `Pipeline` 以外的方式：先對整份資料 `StandardScaler().fit_transform` 再交叉驗證——分數幾乎一樣，但為什麼這也是一種（輕微的）洩漏？
2. 把 R、F、M 換成 `pd.qcut` 的 1～3 分，分數會變嗎？
3. 找一個「預測當下拿不到」的真實例子（例如用「出院日期」預測住院天數）。

---
## 🔑 通關密語
　你已經會用固定的秤比較特徵，也知道分數好到不像話時該做什麼。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🧭 L09 沒有標準答案的學習：PCA 與 K-means** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.2.0-rc.1/notebooks/L09_pca_kmeans.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.2.0-rc.1/